# Vergleichende Analyse des Token-Längen-Experiments (256 vs. 500 vs. 1000 Tokens)

Dieses Notebook analysiert und visualisiert den Einfluss unterschiedlicher maximaler Token-Sequenzlängen (**256**, **500**, **1000**) auf:
1. **Metrik-Training (BiLSTM Regressor)**
2. **SFT Fine-Tuning (mBART-50)**
3. **DPO Alignment (Direct Preference Optimization)**

---

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.size"] = 11
plt.rcParams["figure.titlesize"] = 14

SUMMARY_CSV = "../../../results/evaluation/token_length_comparison_summary.csv"
DETAILS_CSV = "../../../results/evaluation/token_length_comparison_detailed.csv"

# Fallback paths if executed from project root
if not os.path.exists(SUMMARY_CSV):
    SUMMARY_CSV = "results/evaluation/token_length_comparison_summary.csv"
    DETAILS_CSV = "results/evaluation/token_length_comparison_detailed.csv"

## 1. Übersichtstabelle der aggregierten Evaluationsergebnisse

In [ ]:
if os.path.exists(SUMMARY_CSV):
    df_summary = pd.read_csv(SUMMARY_CSV)
    display(df_summary)
else:
    print(f"Evaluationsdatei {SUMMARY_CSV} noch nicht vorhanden. Bitte zuerst die Evaluation ausführen.")
    # Dummy dataframe for notebook structure demonstration
    df_summary = pd.DataFrame({
        'model': ['sft_len256', 'sft_len500', 'sft_len1000', 'dpo_len256', 'dpo_len500', 'dpo_len1000'],
        'max_len': [256, 500, 1000, 256, 500, 1000],
        'r_style_mean': [0.72, 0.74, 0.75, 0.81, 0.84, 0.85],
        'r_sem_as_mean': [0.86, 0.88, 0.89, 0.84, 0.87, 0.88],
        'sim_ref_mean': [0.78, 0.81, 0.83, 0.80, 0.83, 0.85],
        'composite_reward_mean': [0.79, 0.81, 0.82, 0.825, 0.855, 0.865],
        'bleu_mean': [0.18, 0.21, 0.23, 0.20, 0.23, 0.25],
        'rougeL_f1_mean': [0.35, 0.38, 0.40, 0.37, 0.41, 0.43],
        'avg_gen_tokens': [180, 310, 420, 175, 295, 410],
        'compression_ratio_mean': [0.55, 0.65, 0.72, 0.52, 0.62, 0.70],
        'truncation_rate': [0.28, 0.08, 0.01, 0.25, 0.06, 0.00]
    })
    display(df_summary)

## 2. Kernmetriken im Vergleich (Style, Semantik & Composite Reward)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df_summary['stage'] = df_summary['model'].apply(lambda x: 'SFT' if 'sft' in x else 'DPO')

# 1. Style Simplicity Score
sns.barplot(data=df_summary, x='max_len', y='r_style_mean', hue='stage', ax=axes[0], palette=['#4A90E2', '#50E3C2'])
axes[0].set_title("Einfachheits-Score (Style Reward)")
axes[0].set_xlabel("Maximale Token-Länge")
axes[0].set_ylabel("Mittlerer Simplicity Score")
axes[0].set_ylim(0, 1.0)

# 2. Semantic Preservation to AS
sns.barplot(data=df_summary, x='max_len', y='r_sem_as_mean', hue='stage', ax=axes[1], palette=['#4A90E2', '#50E3C2'])
axes[1].set_title("Semantische Erhaltung (SBERT zu AS)")
axes[1].set_xlabel("Maximale Token-Länge")
axes[1].set_ylabel("Mittlere SBERT-Ähnlichkeit")
axes[1].set_ylim(0, 1.0)

# 3. Composite Reward
sns.barplot(data=df_summary, x='max_len', y='composite_reward_mean', hue='stage', ax=axes[2], palette=['#4A90E2', '#50E3C2'])
axes[2].set_title("Composite Reward (50% Style + 50% Semantik)")
axes[2].set_xlabel("Maximale Token-Länge")
axes[2].set_ylabel("Mittlerer Composite Reward")
axes[2].set_ylim(0, 1.0)

plt.tight_layout()
plt.show()

## 3. Längen- und Truncation-Analyse (Kürzungsrate vs. Satzabbrüche)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Truncation Rate
sns.barplot(data=df_summary, x='max_len', y='truncation_rate', hue='stage', ax=axes[0], palette=['#FF6B6B', '#E74C3C'])
axes[0].set_title("Truncation-Rate (Unvollständige Satzenden / Abbrüche)")
axes[0].set_xlabel("Maximale Token-Länge")
axes[0].set_ylabel("Anteil abgeschnittener Texte")
axes[0].set_ylim(0, 0.5)

# 2. Compression Ratio
sns.barplot(data=df_summary, x='max_len', y='compression_ratio_mean', hue='stage', ax=axes[1], palette=['#3498DB', '#2ECC71'])
axes[1].set_title("Kompressionsrate (Generierte Tokens / Quell-Tokens)")
axes[1].set_xlabel("Maximale Token-Länge")
axes[1].set_ylabel("Mittleres Längenverhältnis")
axes[1].set_ylim(0, 1.0)

plt.tight_layout()
plt.show()

## 4. Stratifizierte Analyse nach Textlängen-Klassen der Inputs
Untersuchung, wie sich Modelle auf kurzen (< 200 Tokens), mittleren (200-450 Tokens) und langen (> 450 Tokens) Artikeln verhalten.

In [ ]:
if os.path.exists(DETAILS_CSV):
    df_details = pd.read_csv(DETAILS_CSV)
    
    def categorize_length(src_tokens):
        if src_tokens < 200:
            return 'Kurz (< 200)'
        elif src_tokens <= 450:
            return 'Mittel (200-450)'
        else:
            return 'Lang (> 450)'
            
    df_details['length_cat'] = df_details['src_tokens'].apply(categorize_length)
    
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df_details, x='length_cat', y='composite_reward', hue='model',
                order=['Kurz (< 200)', 'Mittel (200-450)', 'Lang (> 450)'])
    plt.title("Composite Reward aufgeteilt nach Quelltext-Länge")
    plt.xlabel("Längenkategorie des Quelltextes (Alltagssprache)")
    plt.ylabel("Composite Reward")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print("Detailergebnisse nicht gefunden. Führen Sie '5_run_full_evaluation.sh' aus.")

## 5. Qualitativer Text-Vergleich (Side-by-Side)
Vergleich ausgewählter vereinfachter Übersetzungen zwischen 256, 500 und 1000 Token-Modellen.

In [ ]:
if os.path.exists(DETAILS_CSV):
    df_details = pd.read_csv(DETAILS_CSV)
    unique_texts = df_details['as_text'].unique()
    
    if len(unique_texts) > 0:
        sample_text = unique_texts[0]
        sub_df = df_details[df_details['as_text'] == sample_text]
        
        print("=== ORIGINALER QUELLTEXT (Alltagssprache) ===")
        print(sample_text[:400] + ("..." if len(sample_text) > 400 else ""))
        print("\n=== REFERENZTEXT (Leichte Sprache) ===")
        print(sub_df['ls_ref_text'].iloc[0][:400] + "...")
        print("\n" + "="*60)
        
        for _, row in sub_df.iterrows():
            print(f"\nModell: {row['model']} | Tokens: {row['gen_tokens']} | Reward: {row['composite_reward']:.4f} | Truncated: {row['is_truncated']}")
            print(f"Übersetzung: {row['generated_text'][:350]}...")
else:
    print("Keine Detailergebnisse zur qualitativen Ansicht verfügbar.")